# Workout Pose Classifier v6 (MOVIT): 5 kelas, GOOD/BAD + alasan

**Pipeline:** video → MediaPipe 33 keypoint (2D + 3D world) → window 30 frame → fitur v6 → model kelas (Random Forest) → aturan biomekanik GOOD/BAD yang ambangnya **dikalibrasi dari data** (+ model kualitas ML kalau terbukti membantu)

## Yang baru di v6

1. **Label good/bad QEVD otomatis dari kolom `labels`.** Kolom itu sudah berisi catatan kesalahan per video (misalnya `hips too low`, `shallow`, `no obvious issue`). Hasilnya 217 good, 986 bad, 63 invalid. Label manual di kolom `kualitas` tetap **selalu menang**, jadi temanmu cukup mengecek file `outputs/qevd_label_otomatis.csv`, tidak perlu melabel dari nol.
2. **Video invalid dibuang** (`waiting position`, `not visible`, `head moving in an out of frame`). Isinya bukan gerakan yang benar-benar dilakukan, jadi cuma jadi noise untuk model kelas.
3. **Bobot per video.** Di v5, 1 video multiview (rata-rata ±28 window) bobotnya ±5× video QEVD (±5–6 window), dan multiview menyumbang 69% window kelas Pushup. Akibatnya model belajar "Pushup = gaya multiview". Itu salah satu penyebab Pushup QEVD cuma benar 39%. Sekarang tiap video bobotnya sama, dan tiap kelas juga seimbang.
4. **Fitur v6** (di `movit_fitur.py`):
   - sudut kiri/kanan disimpan sebagai pasangan (min, max), jadi tidak peduli sisi mana yang menghadap kamera;
   - offset **bertanda** pinggul dan kepala, jadi bisa membedakan *pinggul turun* dari *pinggul terlalu tinggi*, juga *menunduk* dari *mendongak*;
   - lebar tangan/siku, arah lengan bawah;
   - kecepatan 2D untuk mendeteksi kaki atau tangan yang bergeser.
5. **Aturan GOOD/BAD per kelas dengan ambang hasil kalibrasi.** Ambang tiap aturan dipilih dari tag QEVD (ROC/Youden) memakai split train, lalu diuji di test. Aturan yang lemah di data dilaporkan apa adanya.
6. **Model kualitas ML per kelas.** Hanya dipakai kalau di data test terbukti menaikkan akurasi dibanding aturan saja.
7. **Satu file paket** `movit_model_v6.pkl` (model kelas + aturan + model kualitas) yang langsung dipakai `realtime_workout.py`.

**Cache keypoint v5 (`keypoints/*__3d_15fps.npy`) dipakai ulang**, jadi tidak perlu menjalankan MediaPipe lagi.

Catatan jujur:
- QEVD tidak punya ID orang, jadi orang yang sama bisa muncul di train dan test. Angka QEVD bisa lebih optimis dari kenyataan.
- Label otomatis berasal dari tag dataset, bukan dicek manusia. Pemetaan tag → good/bad ada di bagian 3 dan bisa diubah.

## Struktur folder

```
MOVIT/ml/
├── src/features/
│   ├── movit_fitur.py           # modul fitur bersama (dipakai notebook + realtime)
│   ├── movit_model_v6.pkl       # <- disimpan di sini (bagian 10), satu folder dengan movit_fitur.py
│   └── aturan_v6.json
├── notebook/                    # (contoh) workout_pose_classifier_v6.ipynb
└── Dataset/                     # CSV, Videos/, keypoints/, outputs/
```
`movit_fitur.py` dicari otomatis (`src/features`, `ml/src/features`, `features`, `src`, ... dari folder notebook dan folder Dataset). Kalau tetap tidak ketemu, isi `SRC_DIR` di bagian 1. Di Colab: taruh di `Project ML/src/features/` di Drive.

## 1. Config

In [ ]:
import importlib, json, re, shutil, sys, warnings
from pathlib import Path
from collections import Counter

import joblib
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, classification_report,
                             confusion_matrix, roc_auc_score, roc_curve)

warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

# ---- path ----
ROOT_KANDIDAT = [
    Path("/content/drive/MyDrive/Project ML/Dataset"),              # Colab + Google Drive
    Path(r"C:\Users\hanse\OneDrive\Documents\MOVIT\Dataset"),      # laptop (OneDrive)
    Path(r"C:\Users\hanse\OneDrive\Documents\MOVIT\ml\Dataset"),
    Path.cwd(),                                                     # folder tempat notebook ini dibuka
    *[b / sub for b in (Path.cwd().parent, Path.cwd().parent.parent, Path.cwd().parent.parent.parent)
      for sub in ("Dataset", "ml/Dataset")],                        # mis. MOVIT/ml/notebook -> MOVIT/ml/Dataset
    Path("dataset_pose"),
]
ROOT = next((p for p in ROOT_KANDIDAT if (p / "dataset_index_named.csv").exists()), None)
if ROOT is None:
    raise FileNotFoundError("dataset_index_named.csv tidak ketemu di folder-folder ini:\n  "
                            + "\n  ".join(map(str, ROOT_KANDIDAT))
                            + "\nTaruh CSV-nya di salah satu folder itu, atau tambahkan folder Dataset kamu di ROOT_KANDIDAT.")
CSV_PATH = ROOT / "dataset_index_named.csv"

# ---- modul bersama (dipakai juga oleh realtime_workout.py) ----
SRC_DIR = None   # isi manual kalau tidak ketemu, mis. Path(r"C:\Users\hanse\OneDrive\Documents\MOVIT\ml\src\features")
_basis = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent, ROOT, ROOT.parent, ROOT.parent.parent]
_sub = ("src/features", "ml/src/features", "features", "src", "")
CARI_SRC = ([Path(SRC_DIR)] if SRC_DIR else []) + [
    Path(r"C:\Users\hanse\OneDrive\Documents\MOVIT\ml\src\features")] + [b / s for b in _basis for s in _sub]
for d in CARI_SRC:
    if (d / "movit_fitur.py").exists():
        sys.path.insert(0, str(d))
        break
else:
    raise FileNotFoundError("movit_fitur.py tidak ketemu. Isi SRC_DIR di atas dengan foldernya. Sudah dicari di:\n  "
                            + "\n  ".join(map(str, dict.fromkeys(CARI_SRC))))
import movit_fitur as mf
importlib.reload(mf)
from movit_fitur import (KELAS, NAMA_TAMPIL, KELUARGA_PUSHUP, TARGET_FPS, WINDOW, STRIDE, MAX_MISSING,
                         FITUR_VERSI, bangun_fitur, nama_fitur, metrik_window, metrik_satu, cek_aturan,
                         salin_aturan, pesan)

ALIAS = {"push_up": "Pushup", "push-up": "Pushup"}      # nama alternatif di CSV
SUMBER = ("qevd", "multiview", "pushup")               # = 3 folder dataset


def cari_folder(parent, nama):
    """cari subfolder `nama` tanpa peduli huruf besar/kecil; kalau belum ada pakai nama aslinya"""
    parent = Path(parent)
    if parent.exists():
        for p in parent.iterdir():
            if p.is_dir() and p.name.lower() == nama.lower():
                return p
    return parent / nama


VIDEO_ROOT = cari_folder(ROOT, "Videos")
KP, OUT_DIR = ROOT / "keypoints", ROOT / "outputs"
FOLDER_VIDEO = {"qevd": "Qevd", "multiview": "SevenFitness", "pushup": "Pushup"}
SRC_DIRS = {s: cari_folder(VIDEO_ROOT, FOLDER_VIDEO[s]) for s in SUMBER}
# model final disimpan di folder yang sama dengan movit_fitur.py (mis. MOVIT/ml/src/features)
MODEL_DIR = Path(mf.__file__).resolve().parent
for p in [KP, OUT_DIR, MODEL_DIR, *SRC_DIRS.values()]:
    p.mkdir(parents=True, exist_ok=True)

STATUS_DIPAKAI = {"ok"}          # tambah "review" kalau mau ikut pakai video yang NaN-nya banyak

# ---- label kualitas ----
PAKAI_LABEL_OTOMATIS_QEVD = True   # good/bad QEVD dari kolom `labels`; label manual (kolom kualitas) selalu menang
KELUARKAN_VIDEO_INVALID = True     # buang video bertag waiting position / not visible / kepala keluar frame

# ---- kalibrasi aturan & penilaian ----
MIN_VIDEO_KALIBRASI = 8      # minimal video good (dan video bertag) agar satu aturan dikalibrasi
FPR_TOTAL = 0.10             # jatah alarm palsu awal per kelas (window GOOD), dibagi rata ke semua aturan kelas itu
CARI_FPR = (0.05, 0.10, 0.15, 0.20, 0.30, 0.40, 0.50)   # bagian 8 memilih jatah terbaik per kelas dari data train
TOLERANSI = {"siku": 5.0, "lutut": 5.0, "tubuh": 5.0, "dev": 0.02, "lebar": 0.10, "tangan_vs": 0.10, "gerak": 0.01}
AUC_MIN = 0.65               # aturan yang tidak bisa membedakan video bertag dari video good -> lemah
NONAKTIFKAN_ATURAN_LEMAH = True
AUC_KUAT = 0.80              # aturan sekuat ini boleh memakai ambang "titik terbaik" (lebih banyak bad tertangkap)
FPR_KUAT_MAKS = 0.15         # ... asal video good yang ikut terpicu aturan itu maksimal 15%
PROPORSI_WINDOW_BAD = 0.5    # video BAD kalau SATU aturan yang sama terpicu di >= 50% window-nya
MIN_VIDEO_ML_KUALITAS = 15   # minimal video good DAN bad (train) agar model kualitas ML dicoba untuk kelas itu

print("root:", ROOT)
print("modul fitur:", mf.__file__, "| versi:", FITUR_VERSI)
print("folder video:", {k: str(v) for k, v in SRC_DIRS.items()})
print("model disimpan ke:", MODEL_DIR)

## 2. Fungsi ekstraksi keypoint & window

Fitur dan aturan sekarang ada di `movit_fitur.py`. Bagian ini hanya mengurus video → keypoint → window, sama seperti v5.

In [ ]:
def norm_kelas(name):
    k = str(name).lower().strip().replace(" ", "_")
    for kn in KELAS:
        if kn.lower() == k:
            return kn
    return ALIAS.get(k)


def ambil_frame(i, fps, target=TARGET_FPS):
    return int(i * target / fps) != int((i - 1) * target / fps)


def video_to_keypoints(path, fps_src=None):
    """video -> (T, 33, 5): [x2d, y2d, x3d, y3d, z3d]; frame tanpa pose = NaN."""
    import cv2, mediapipe as mp
    frames, i = [], 0
    cap = cv2.VideoCapture(str(path))
    fps = fps_src or cap.get(cv2.CAP_PROP_FPS) or 30.0
    with mp.solutions.pose.Pose(min_detection_confidence=0.5, model_complexity=1) as pose:
        while cap.isOpened():
            ok, frame = cap.read()
            if not ok:
                break
            i += 1
            if not ambil_frame(i, fps):
                continue
            res = pose.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            if res.pose_landmarks and res.pose_world_landmarks:
                lm2d, lm3d = res.pose_landmarks.landmark, res.pose_world_landmarks.landmark
                frames.append([[lm2d[j].x, lm2d[j].y, lm3d[j].x, lm3d[j].y, lm3d[j].z] for j in range(33)])
            else:
                frames.append(np.full((33, 5), np.nan))
    cap.release()
    return np.array(frames, dtype=np.float32).reshape(-1, 33, 5)


def make_windows(kp):
    out = []
    D = kp.shape[-1]
    for s in range(0, len(kp) - WINDOW + 1, STRIDE):
        w = kp[s:s + WINDOW]
        miss = np.isnan(w).any(axis=(1, 2))
        if miss.mean() > MAX_MISSING or miss.all():
            continue
        if miss.any():
            w = (pd.DataFrame(w.reshape(WINDOW, -1)).interpolate(limit_direction="both")
                 .to_numpy().reshape(WINDOW, 33, D))
        out.append((s, w.astype(np.float32)))
    return out

## 3. Baca CSV + label good/bad otomatis untuk QEVD

Tiap video QEVD punya tag di kolom `labels`. Pemetaannya:
- **invalid**: video tidak menunjukkan gerakan yang benar (`waiting position`, `not visible`, `head moving in an out of frame`), jadi dibuang.
- **bad**: ada minimal satu tag kesalahan untuk kelas itu (tabel `TAG_BAD`).
- **good**: tidak ada tag kesalahan dan ada tag baik (`average`, `no obvious issue`, `deep`, `90 degrees`).
- **unknown**: tag-nya netral saja (misalnya `wide`, `stopping early`), jadi tidak dipakai untuk kualitas.

Hasilnya diekspor ke `outputs/qevd_label_otomatis.csv` supaya temanmu bisa mengecek. Kalau temanmu mengisi kolom `kualitas` di CSV utama dengan good/bad, isian itu yang dipakai.

In [ ]:
TAG_BAD = {
    "High_Plank": {"looking up", "hips too high", "hips too low", "rounded shoulders", "knees on floor",
                   "knees bent", "elbows bent", "moving feet", "moving arms"},
    "Elbow_Plank": {"legs and hips on the floor", "butt sinking", "arms facing inward", "on knees",
                    "shoulders shrugged up toward the ears", "head up", "high hips", "arms too wide", "head down",
                    "shaking", "arms facing inward and legs and hips on the floor", "arms facing inward and on knees"},
    "Pushup": {"shallow", "insufficient", "very wide", "butt sinking", "elevated hips", "child pose cheat",
               "legs on the floor", "sloppy", "side to side", "fast", "moving head", "head down"},
    "Pushup_on_Knees": {"shallow", "insufficient", "very wide", "elevated hips", "butt sinking", "moving head",
                        "head down", "sloppy"},
    "Side_Plank": {"high hips", "low hips", "drooping head", "leaning forward", "both knees on floor",
                   "one knee on floor", "losing balance"},
}
TAG_BAIK = {"average", "no obvious issue", "deep", "90 degrees"}
TAG_INVALID = {"waiting position", "not visible", "head moving in an out of frame"}


def parse_tags(s):
    if not isinstance(s, str) or not s.strip():
        return tuple()
    return tuple(t.split(" - ", 1)[-1].strip() for t in s.split("|") if t.strip())


def label_otomatis(kelas, tags):
    t = set(tags)
    if t & TAG_INVALID:
        return "invalid", ", ".join(sorted(t & TAG_INVALID))
    salah = t & TAG_BAD.get(kelas, set())
    if salah:
        return "bad", ", ".join(sorted(salah))
    if t & TAG_BAIK:
        return "good", ", ".join(sorted(t & TAG_BAIK))
    return "unknown", ", ".join(sorted(t))


NAMA_KOLOM = ("file", "nama_file", "filename", "video", "video_name")
idx = pd.read_csv(CSV_PATH, dtype={c: str for c in NAMA_KOLOM})
butuh = {"id", "sumber", "kelas", "kualitas", "split", "status"}
assert "file" in idx.columns, "CSV ini belum punya kolom `file`; pakai dataset_index_named.csv"
assert butuh <= set(idx.columns), f"kolom CSV kurang: {butuh - set(idx.columns)}"
print("baris di CSV:", len(idx))

idx["sumber"] = idx["sumber"].astype(str).str.lower().str.strip()
idx["kelas_n"] = idx["kelas"].map(norm_kelas)
idx["kualitas"] = idx["kualitas"].fillna("unknown").astype(str).str.lower().str.strip()
idx["view"] = idx["view"].fillna("") if "view" in idx.columns else ""
idx["tags"] = idx["labels"].map(parse_tags) if "labels" in idx.columns else [tuple()] * len(idx)

alasan = Counter()
for _, r in idx[idx.kelas_n.isna()].iterrows():
    alasan[f"kelas '{r.kelas}' bukan 5 kelas final"] += 1
for _, r in idx[~idx.sumber.isin(SUMBER)].iterrows():
    alasan[f"sumber '{r.sumber}' tidak dikenal"] += 1
for _, r in idx[~idx.status.isin(STATUS_DIPAKAI)].iterrows():
    alasan[f"status '{r.status}' tidak dipakai"] += 1
keep = idx.kelas_n.notna() & idx.sumber.isin(SUMBER) & idx.status.isin(STATUS_DIPAKAI)
if alasan:
    print("\ndilewati:")
    for k, v in alasan.items():
        print(f"  {v:4d}  {k}")
idx = idx[keep].copy()

# ---- label kualitas final: manual > otomatis (QEVD) > unknown ----
auto = idx.apply(lambda r: label_otomatis(r.kelas_n, r.tags) if r.sumber == "qevd" else ("", ""), axis=1)
idx["kualitas_otomatis"] = [a for a, _ in auto]
idx["alasan_otomatis"] = [b for _, b in auto]
idx["invalid"] = idx.kualitas_otomatis.eq("invalid") & KELUARKAN_VIDEO_INVALID
manual = idx.kualitas.isin(["good", "bad"])
idx["sumber_label"] = np.where(manual, "manual",
                               np.where(idx.kualitas_otomatis.isin(["good", "bad"]) & PAKAI_LABEL_OTOMATIS_QEVD,
                                        "otomatis", "-"))
idx["kualitas_final"] = np.where(manual, idx.kualitas,
                                 np.where(idx.sumber_label == "otomatis", idx.kualitas_otomatis, "unknown"))

bad_q = idx[idx.sumber.isin(["multiview", "pushup"]) & ~manual]
assert bad_q.empty, f"{len(bad_q)} baris multiview/pushup tanpa label good/bad, contoh: {bad_q.id.head(3).tolist()}"
assert idx.split.isin(["train", "test"]).all(), "kolom split harus train / test"
assert idx.id.is_unique, "kolom id harus unik"

kol_nama = next((c for c in NAMA_KOLOM if c in idx.columns), None)
idx["nama_vid"] = (idx[kol_nama].fillna("").astype(str).map(lambda x: Path(x).stem.strip().lower())
                   if kol_nama else "")

subj = idx["subjek"] if "subjek" in idx.columns else pd.Series(np.nan, index=idx.index)
idx["group"] = np.where((idx.sumber == "multiview") & subj.notna(),
                        "multiview_s" + subj.fillna(0).astype(int).astype(str), idx.id)
idx["fps_src"] = pd.to_numeric(idx["fps_asli"], errors="coerce") if "fps_asli" in idx.columns else np.nan
idx["cache_id"] = idx.id + f"__3d_{TARGET_FPS}fps"      # sama dengan cache v5

# ---- file untuk dicek temanmu ----
rev = idx[idx.sumber == "qevd"][["id", "file", "kelas_n", "split", "labels", "kualitas",
                                 "kualitas_otomatis", "alasan_otomatis"]].rename(
    columns={"kelas_n": "kelas", "kualitas": "kualitas_manual"})
rev.to_csv(OUT_DIR / "qevd_label_otomatis.csv", index=False)
print(f"\nfile review label: {OUT_DIR / 'qevd_label_otomatis.csv'}")

print(f"\nvideo dipakai: {len(idx)} | invalid dibuang: {int(idx.invalid.sum())}")
print("\nlabel kualitas final (video):")
display(pd.crosstab([idx.sumber, idx.kelas_n], idx.kualitas_final, margins=True))
print("asal label:", idx.sumber_label.value_counts().to_dict())
display(pd.crosstab(idx.sumber, idx.split))

## 4. (Opsional) Pilah video ke 3 folder sumber

Kalau semua video masih ada di satu folder campur, isi `DUMP_DIR` lalu run cell ini. Kalau video sudah ada di 3 folder, biarkan `DUMP_DIR = None`.

In [ ]:
DUMP_DIR = None      # contoh: ROOT / "semua_video"
PINDAH = False       # False = salin (aman), True = pindahkan
VIDEO_EXT = {".mp4", ".mov", ".avi", ".mkv"}

if DUMP_DIR is not None:
    peta = {}
    for r in idx.itertuples():
        peta[r.id.lower()] = r.sumber
        if r.nama_vid:
            peta[r.nama_vid] = r.sumber
    n = Counter()
    for p in tqdm(sorted(Path(DUMP_DIR).rglob("*")), desc="memilah"):
        if p.suffix.lower() not in VIDEO_EXT or p.stem.lower() not in peta:
            continue
        tujuan = SRC_DIRS[peta[p.stem.lower()]] / p.name
        if not tujuan.exists():
            (shutil.move if PINDAH else shutil.copy2)(p, tujuan)
        n[peta[p.stem.lower()]] += 1
    print("dipilah per sumber:", dict(n))
else:
    print("DUMP_DIR kosong, bagian ini dilewati")

## 5. Cocokkan CSV dengan file video

Baris yang videonya tidak ada **dan** belum punya cache keypoint dilaporkan dan dilewati.

In [ ]:
def index_folder(d):
    out = {}
    if Path(d).exists():
        for p in sorted(Path(d).rglob("*")):
            if p.suffix.lower() in VIDEO_EXT:
                out.setdefault(p.stem.lower(), p)
    return out


def kunci(r):
    ks = [r.nama_vid] if r.nama_vid else []
    return ks + [r.id.lower(), r.id.split("_", 1)[-1].lower()]


folder_idx = {s: index_folder(SRC_DIRS[s]) for s in SUMBER}
paths, ada_cache, hilang = [], [], {s: [] for s in SUMBER}
for r in idx.itertuples():
    p = next((folder_idx[r.sumber][k] for k in kunci(r) if k in folder_idx[r.sumber]), None)
    c = (KP / f"{r.cache_id}.npy").exists()
    paths.append(p)
    ada_cache.append(c)
    if p is None and not c:
        hilang[r.sumber].append(r.nama_vid or r.id)
idx["path"] = paths
idx["ada_cache"] = ada_cache

ITEMS = idx[idx.path.notna() | idx.ada_cache].copy()
print(f"siap diproses: {len(ITEMS)} dari {len(idx)} video")
display(pd.crosstab(ITEMS.sumber, ITEMS.split, margins=True))
for s, v in hilang.items():
    if v:
        print(f"[{s}] {len(v)} video di CSV tidak ketemu di {SRC_DIRS[s]} (contoh nama: {v[:3]})")
assert len(ITEMS), "tidak ada video yang ketemu. Cek nama file video cocok dengan kolom `file` di CSV"

## 6. Ekstraksi video → keypoint 3D → window

Format cache sama dengan v5 (`keypoints/<id>__3d_15fps.npy`), jadi video yang sudah diekstrak langsung dilewati.

In [ ]:
import subprocess

WORKER = ROOT / "_extract_worker_3d.py"
WORKER_CODE = r"""
import sys, json
from pathlib import Path
import numpy as np

def main():
    job = json.load(open(sys.argv[1], encoding="utf-8"))
    kp_dir = Path(job["kp_dir"])
    T = float(job["target_fps"])
    import cv2, mediapipe as mp
    for it in job["items"]:
        out = kp_dir / (it["id"] + ".npy")
        if out.exists():
            continue
        (kp_dir / job["progress"]).write_text(it["id"], encoding="utf-8")
        frames, i = [], 0
        cap = cv2.VideoCapture(it["path"])
        fps = it["fps"] or cap.get(cv2.CAP_PROP_FPS) or 30.0
        with mp.solutions.pose.Pose(min_detection_confidence=0.5, model_complexity=1) as pose:
            while cap.isOpened():
                ok, frame = cap.read()
                if not ok:
                    break
                i += 1
                if int(i * T / fps) == int((i - 1) * T / fps):
                    continue
                res = pose.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
                if res.pose_landmarks and res.pose_world_landmarks:
                    lm2d = res.pose_landmarks.landmark
                    lm3d = res.pose_world_landmarks.landmark
                    frames.append([[lm2d[j].x, lm2d[j].y, lm3d[j].x, lm3d[j].y, lm3d[j].z] for j in range(33)])
                else:
                    frames.append(np.full((33, 5), np.nan))
        cap.release()
        np.save(out, np.array(frames, dtype=np.float32).reshape(-1, 33, 5))
        print("OK", it["id"], len(frames), flush=True)

main()
"""
WORKER.write_text(WORKER_CODE, encoding="utf-8")
print("worker:", WORKER)

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from threading import Lock

N_WORKERS = 3     # jumlah proses MediaPipe paralel

GAGAL_FILE = KP / "_gagal_3d.txt"
gagal = {x for x in GAGAL_FILE.read_text(encoding="utf-8").split("\n") if x} if GAGAL_FILE.exists() else set()
lock = Lock()


def belum(items):
    return [r for r in items
            if r.cache_id not in gagal and not (KP / f"{r.cache_id}.npy").exists() and r.path is not None]


todo0 = belum(list(ITEMS.itertuples()))
print(f"sudah ada cache 3D: {len(ITEMS) - len(todo0) - len(gagal)} | perlu diekstrak: {len(todo0)} | "
      f"dilewati (gagal sebelumnya): {len(gagal)} | worker paralel: {N_WORKERS}")
bar = tqdm(total=len(todo0), desc="ekstraksi 3D")


def jalur(k, items):
    prog, logf = KP / f"_sedang_3d_{k}.txt", KP / f"_worker_3d_{k}.log"
    for _ in range(len(items) + 5):
        todo = belum(items)
        if not todo:
            return
        job = KP / f"_job_3d_{k}.json"
        job.write_text(json.dumps(dict(
            kp_dir=str(KP), progress=prog.name, target_fps=TARGET_FPS,
            items=[dict(id=r.cache_id, path=str(r.path), fps=None if pd.isna(r.fps_src) else float(r.fps_src))
                   for r in todo])), encoding="utf-8")
        prog.unlink(missing_ok=True)
        with open(logf, "a", encoding="utf-8") as log:
            p = subprocess.Popen([sys.executable, str(WORKER), str(job)], stdout=subprocess.PIPE, stderr=log, text=True)
            for line in p.stdout:
                if line.startswith("OK"):
                    bar.update(1)
            p.wait()
        if p.returncode == 0:
            continue
        sedang = prog.read_text(encoding="utf-8").strip() if prog.exists() else ""
        if not sedang:
            tail = logf.read_text(encoding="utf-8", errors="ignore").splitlines()[-15:]
            raise RuntimeError("proses ekstraksi berhenti sebelum video pertama:\n" + "\n".join(tail))
        with lock:
            nama = idx.loc[idx.cache_id == sedang, "nama_vid"].iloc[0] or sedang
            print(f"\n[worker {k}] crash di video {sedang} ({nama}), kode {p.returncode}. Dilewati, lanjut.")
            gagal.add(sedang)
            GAGAL_FILE.write_text("\n".join(sorted(gagal)), encoding="utf-8")
    raise RuntimeError(f"worker {k}: terlalu banyak percobaan ulang, cek {logf}")


chunks = [todo0[k::N_WORKERS] for k in range(N_WORKERS)]
with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
    for f in [ex.submit(jalur, k, c) for k, c in enumerate(chunks) if c]:
        f.result()
bar.close()

ok = sum((KP / f"{r.cache_id}.npy").exists() for r in ITEMS.itertuples())
print(f"\nselesai. keypoint 3D tersedia: {ok}/{len(ITEMS)} video | gagal/dilewati: {len(gagal)}")

In [ ]:
Xs, rows, kosong, n_invalid = [], [], [], 0
for r in tqdm(list(ITEMS.itertuples()), desc="window"):
    if r.invalid:
        n_invalid += 1
        continue
    cache = KP / f"{r.cache_id}.npy"
    if r.cache_id in gagal or not cache.exists():
        continue
    kp = np.load(cache)[:, :33, :5].astype(np.float32)
    ws = make_windows(kp)
    if not ws:
        kosong.append(r.id)
    for s, w in ws:
        Xs.append(w)
        rows.append(dict(video_id=r.id, mulai_frame=s, sumber=r.sumber, kelas=r.kelas_n, kualitas=r.kualitas_final,
                         sumber_label=r.sumber_label, view=r.view, split=r.split, group=r.group, tags=r.tags))

print(f"video invalid dilewati: {n_invalid}")
if kosong:
    print(f"{len(kosong)} video tidak menghasilkan window (terlalu pendek / pose hilang), contoh:", kosong[:3])
assert Xs, "tidak ada window sama sekali"

X_all = np.stack(Xs).astype(np.float32)          # (N, 30, 33, 5)
META = pd.DataFrame(rows)
assert len(META) == len(X_all) and META.kelas.isin(KELAS).all()

# satu baris per video (dipakai untuk evaluasi video-level)
VID = META.groupby("video_id").agg(kelas=("kelas", "first"), sumber=("sumber", "first"),
                                   kualitas=("kualitas", "first"), split=("split", "first"),
                                   tags=("tags", "first"), n_window=("kelas", "size"))

print("\nTOTAL window:", X_all.shape, "| video:", len(VID))
display(pd.crosstab([META.sumber, META.kelas], META.kualitas, margins=True))
print("orang/video yang kepecah ke train & test:", int((META.groupby("group").split.nunique() > 1).sum()), "(harus 0)")

## 7. Model kelas (gerakan apa)

**Bobot per video:** setiap video total bobotnya sama (`1 / jumlah window`), lalu setiap kelas total bobotnya juga sama. Dengan begitu video multiview yang panjang tidak lagi mendominasi kelas Pushup.

In [ ]:
F_ALL = bangun_fitur(X_all)
FEAT_COLS = nama_fitur()
M_ALL = metrik_window(X_all)          # metrik aturan per window (dipakai bagian 8-11)
Y_KELAS = META.kelas.map({k: i for i, k in enumerate(KELAS)}).to_numpy()
print(f"fitur {FITUR_VERSI}: {F_ALL.shape[1]} kolom")


def bobot_sampel(meta, kolom):
    """tiap video total bobot sama, lalu tiap nilai `kolom` (kelas / kualitas) total bobot sama"""
    n = meta.groupby("video_id")["video_id"].transform("size").to_numpy(float)
    w = pd.Series(1.0 / n, index=meta.index)
    w = w / w.groupby(meta[kolom]).transform("sum")
    return (w / w.mean()).to_numpy()


def make_model(max_depth=14, n=400):
    return RandomForestClassifier(n_estimators=n, max_depth=max_depth, min_samples_leaf=3, max_features="sqrt",
                                  n_jobs=-1, random_state=42)


def vote(video_ids, y_true, y_pred):
    t = pd.DataFrame({"v": video_ids, "y": y_true, "p": y_pred})
    return t.groupby("v").agg(y=("y", "first"), p=("p", lambda s: s.mode().iloc[0]))


def evaluate(y, pred, vids, names, title):
    v = vote(vids, y, pred)
    print(f"=== {title} ===")
    print(f"window accuracy: {accuracy_score(y, pred):.3f} ({len(y)} window) | "
          f"video accuracy: {accuracy_score(v.y, v.p):.3f} ({len(v)} video)")
    labels = list(range(len(names)))
    print(classification_report(v.y, v.p, labels=labels, target_names=names, zero_division=0))
    print("confusion matrix (video, baris = asli, kolom = prediksi):")
    print(pd.DataFrame(confusion_matrix(v.y, v.p, labels=labels), index=names, columns=names))
    return v


tr, te = (META.split == "train").to_numpy(), (META.split == "test").to_numpy()
W_KELAS = bobot_sampel(META, "kelas")
m_kelas = make_model().fit(F_ALL[tr], Y_KELAS[tr], sample_weight=W_KELAS[tr])
v_kelas = evaluate(Y_KELAS[te], m_kelas.predict(F_ALL[te]), META.video_id[te].to_numpy(), KELAS, "KELAS, train -> test")

for s in SUMBER:
    ms = te & (META.sumber == s).to_numpy()
    if ms.sum():
        vs = vote(META.video_id[ms].to_numpy(), Y_KELAS[ms], m_kelas.predict(F_ALL[ms]))
        print(f"  video accuracy {s:10s}: {accuracy_score(vs.y, vs.p):.3f} ({len(vs)} video test)")

# ---- diagnosa: video QEVD yang salah, tag apa yang paling sering? ----
vk = v_kelas.join(VID[["sumber", "tags"]])
salah = vk[(vk.y != vk.p) & (vk.sumber == "qevd")]
if len(salah):
    semua = Counter(t for ts in vk[vk.sumber == "qevd"].tags for t in ts)
    kena = Counter(t for ts in salah.tags for t in ts)
    diag = pd.DataFrame({"salah": kena, "total_test": semua}).dropna()
    diag["porsi_salah"] = (diag.salah / diag.total_test).round(2)
    print(f"\nQEVD test salah: {len(salah)} video. Tag yang paling sering ada di video salah:")
    display(diag.sort_values(["salah", "porsi_salah"], ascending=False).head(10))

imp = pd.Series(m_kelas.feature_importances_, index=FEAT_COLS).sort_values(ascending=False)
print("\n10 fitur terpenting:")
display(imp.head(10).round(3))

## 8. Aturan GOOD/BAD: ambang dari batas normal video good

**Kenapa diubah (hasil run pertama v6):** ambang Youden terlalu galak. Di Pushup cuma 1 dari 40 video good yang lolos, terutama karena `PINGGUL_NAIK` dan `KURANG_LURUS`. Ada dua penyebab:
1. Ambang dipilih tanpa memperhatikan berapa banyak video good yang ikut kena.
2. Tiap kelas punya ±8 aturan dan cukup satu yang terpicu untuk membuat BAD, jadi peluang alarm palsu menumpuk.

Cara sekarang:
1. **Ambang = batas normal video good**, dihitung dari **semua window** video good (QEVD otomatis + multiview/pushup manual), sehingga tiap aturan hanya memicu di `FPR_TOTAL / jumlah_aturan` window good. Contoh: 8 aturan, jadi masing-masing ±1.25%.
2. Ditambah **toleransi** (`TOLERANSI`, misalnya 5° untuk sudut) supaya noise kecil MediaPipe tidak memicu.
3. **AUC** = seberapa bisa aturan membedakan video QEVD bertag kesalahannya dari video good. Kalau < `AUC_MIN`, aturan itu **dimatikan**. Kolom `tpr` = porsi video bertag yang tertangkap.
4. Video **BAD** kalau satu aturan yang sama terpicu di ≥ `PROPORSI_WINDOW_BAD` window-nya. Di realtime padanannya: aturan terpicu di 2 dari 3 penilaian terakhir.

**Run kedua:** jatah 10% membuat video good lolos semua, tapi banyak video bad tidak tertangkap (Pushup cuma 8/57). Jadi sekarang jatah alarm palsu (`CARI_FPR`) **dipilih per kelas** dari balanced accuracy di data train.

**Run ketiga:** beberapa aturan sebenarnya kuat (AUC ≥ 0.8), tapi ambangnya masih memakai batas aman yang sama dengan aturan lemah, jadi bad yang tertangkap cuma 14–55%. Sekarang aturan dengan AUC ≥ `AUC_KUAT` memakai **titik terbaik** antara video bertag dan video good, asal video good yang ikut terpicu maksimal `FPR_KUAT_MAKS` (15%). Kolom `fpr_good` = porsi video good yang memicu aturan itu.

Kalibrasi dan pemilihan jatah memakai split train, lalu penilaian GOOD/BAD diuji di split test. Tabel terakhir menunjukkan tag kesalahan mana yang paling sering lolos.

In [ ]:
TAG_ATURAN = {
    "PINGGUL_TURUN": {"hips too low", "butt sinking", "low hips", "legs and hips on the floor",
                      "arms facing inward and legs and hips on the floor"},
    "PINGGUL_NAIK": {"hips too high", "high hips", "elevated hips"},
    "LUTUT_TEKUK": {"knees bent", "knees on floor", "on knees", "arms facing inward and on knees",
                    "both knees on floor", "one knee on floor", "legs on the floor"},
    "KURANG_DALAM": {"shallow", "insufficient"},
    "SIKU_TEKUK": {"elbows bent"},
    "TANGAN_LEBAR": {"very wide"},
    "KEPALA_TURUN": {"head down", "drooping head"},
    "KEPALA_NAIK": {"head up", "looking up"},
    "KAKI_BERGERAK": {"moving feet"},
    "TANGAN_BERGERAK": {"moving arms"},
    "LENGAN_KE_DALAM": {"arms facing inward", "arms facing inward and legs and hips on the floor",
                        "arms facing inward and on knees"},
    "LENGAN_LEBAR": {"arms too wide"},
}

VID_M = pd.DataFrame(M_ALL).groupby(META.video_id.to_numpy()).median()
VID = VID.drop(columns=[c for c in VID_M.columns if c in VID.columns]).join(VID_M)


def toleransi(metrik):
    return next((v for k, v in TOLERANSI.items() if metrik.startswith(k)), 0.0)


def kalibrasi(vid, aturan_awal, fpr_kelas=None):
    """ambang = batas normal window video GOOD (+ toleransi); AUC dari video QEVD bertag vs video good.
    `vid` = tabel per video yang dipakai (mis. hanya split train). fpr_kelas = {kelas: jatah alarm palsu}."""
    aturan, lap = salin_aturan(aturan_awal), []
    vid_w = META.video_id.to_numpy()
    for kelas, daftar in aturan.items():
        dk = vid[vid.kelas == kelas]
        good = dk[dk.kualitas == "good"]
        mw = np.isin(vid_w, good.index.to_numpy())          # window milik video good
        fpr_rule = (fpr_kelas or {}).get(kelas, FPR_TOTAL) / max(1, len(daftar))
        for r in daftar:
            row = dict(kelas=kelas, aturan=r["id"], metrik=r["metrik"], arah=r["arah"], default=r["ambang"],
                       n_good=len(good))
            xw = M_ALL[r["metrik"]][mw]
            xw = xw[np.isfinite(xw)]
            if len(good) < MIN_VIDEO_KALIBRASI or len(xw) == 0:
                lap.append({**row, "status": "video good kurang -> default"})
                continue
            tol = toleransi(r["metrik"])
            if r["arah"] == ">":
                ambang = float(np.quantile(xw, 1 - fpr_rule)) + tol
            else:
                ambang = float(np.quantile(xw, fpr_rule)) - tol
            r["ambang"], r["sumber"] = ambang, "batas normal good"
            row["ambang_baru"] = round(ambang, 3)

            tags = TAG_ATURAN.get(r["id"])
            pos = dk[(dk.sumber == "qevd") & dk.tags.map(lambda t: bool(set(t) & tags))] if tags else dk.iloc[:0]
            xp = pos[r["metrik"]].to_numpy(float)
            xp = xp[np.isfinite(xp)]
            xg = good[r["metrik"]].to_numpy(float)
            xg = xg[np.isfinite(xg)]
            row["n_bertag"] = len(xp)
            if len(xp) >= MIN_VIDEO_KALIBRASI:
                y = np.r_[np.ones(len(xp)), np.zeros(len(xg))]
                x = np.r_[xp, xg]
                auc = roc_auc_score(y, x if r["arah"] == ">" else -x)
                status = "dikalibrasi" + (" (lemah)" if auc < AUC_MIN else "")
                if auc >= AUC_KUAT:
                    # aturan kuat: titik terbaik bertag-vs-good, dengan alarm palsu video good <= FPR_KUAT_MAKS.
                    # hanya boleh MELONGGARKAN ke arah menangkap lebih banyak bad, tidak lebih ketat.
                    sk = x if r["arah"] == ">" else -x
                    f_, t_, th_ = roc_curve(y, sk)
                    okm = np.isfinite(th_) & (f_ <= FPR_KUAT_MAKS)
                    if okm.any():
                        i = int(np.argmax(np.where(okm, t_ - f_, -1)))
                        cand = float(th_[i]) if r["arah"] == ">" else float(-th_[i])
                        if (r["arah"] == ">" and cand < ambang) or (r["arah"] == "<" and cand > ambang):
                            ambang = cand
                            r["ambang"], r["sumber"] = ambang, "titik terbaik (aturan kuat)"
                            row["ambang_baru"] = round(ambang, 3)
                            status = "dikalibrasi (kuat -> titik terbaik)"
                tpr = float(np.mean(xp > ambang) if r["arah"] == ">" else np.mean(xp < ambang))
                fpr_g = float(np.mean(xg > ambang) if r["arah"] == ">" else np.mean(xg < ambang))
                row.update(auc=round(auc, 3), tpr=round(tpr, 2), fpr_good=round(fpr_g, 2))
                if auc < AUC_MIN and NONAKTIFKAN_ATURAN_LEMAH:
                    r["aktif"] = False
                    lap.append({**row, "status": "lemah -> DIMATIKAN"})
                    continue
                lap.append({**row, "status": status})
            else:
                lap.append({**row, "status": "dikalibrasi (tanpa tag, cuma batas normal)"})
    return aturan, pd.DataFrame(lap)




def nilai_aturan_video(mask, aturan):
    """vonis aturan per video untuk window yang dipilih mask."""
    ids = np.where(mask)[0]
    kelas_w, vid_w = META.kelas.to_numpy(), META.video_id.to_numpy()
    bad, alasan = [], []
    for i in ids:
        iss = cek_aturan(metrik_satu(M_ALL, i), kelas_w[i], aturan)
        bad.append(bool(iss))
        alasan.append([a for a, _ in iss])
    t = pd.DataFrame({"v": vid_w[ids], "alasan": alasan})
    out = {}
    for v, g in t.groupby("v"):
        hit = Counter(a for al in g.alasan for a in al)
        aktif = [a for a, n in hit.items() if n / len(g) >= PROPORSI_WINDOW_BAD]
        out[v] = dict(rule_bad=bool(aktif), alasan=aktif)
    return pd.DataFrame.from_dict(out, orient="index")


def ba(y_bad, pred_bad):
    y_bad, pred_bad = np.asarray(y_bad, bool), np.asarray(pred_bad, bool)
    if y_bad.all() or (~y_bad).all():
        return np.nan
    return round(balanced_accuracy_score(y_bad, pred_bad), 3)


LAB = META.kualitas.isin(["good", "bad"]).to_numpy()
VID_TR = VID[VID.split == "train"]

# ---- pilih jatah alarm palsu terbaik per kelas (dihitung di data train saja) ----
skor_fpr = []
for f in CARI_FPR:
    at, _ = kalibrasi(VID_TR, None, {k: f for k in KELAS})
    h = nilai_aturan_video(tr & LAB, at).join(VID[["kelas", "kualitas"]])
    for k in KELAS:
        hk = h[h.kelas == k]
        yb = (hk.kualitas == "bad").to_numpy()
        skor_fpr.append(dict(kelas=k, fpr=f, balanced_acc=ba(yb, hk.rule_bad),
                             good_lolos=float((~hk.rule_bad[~yb]).mean()) if (~yb).any() else np.nan,
                             bad_tertangkap=float(hk.rule_bad[yb].mean()) if yb.any() else np.nan))
skor_fpr = pd.DataFrame(skor_fpr)
FPR_KELAS = {k: float(g.sort_values(["balanced_acc", "fpr"], ascending=[False, True]).fpr.iloc[0])
             for k, g in skor_fpr.dropna(subset=["balanced_acc"]).groupby("kelas")}
print("Balanced accuracy aturan di TRAIN untuk tiap jatah alarm palsu:")
display(skor_fpr.pivot(index="kelas", columns="fpr", values="balanced_acc").round(3))
print("jatah alarm palsu terpilih per kelas:", FPR_KELAS)

ATURAN_TRAIN, lap_train = kalibrasi(VID_TR, None, FPR_KELAS)
print("\nKalibrasi ambang (split train, jatah terpilih):")
display(lap_train)

hasil_te = nilai_aturan_video(te & LAB, ATURAN_TRAIN).join(VID[["kelas", "kualitas", "sumber", "tags"]])

ring = []
for k in KELAS:
    h = hasil_te[hasil_te.kelas == k]
    if not len(h):
        continue
    yb = (h.kualitas == "bad").to_numpy()
    alarm = Counter(a for al in h[~yb].alasan for a in set(al))
    ring.append(dict(kelas=k, video_test=len(h), good=int((~yb).sum()), bad=int(yb.sum()),
                     balanced_acc=ba(yb, h.rule_bad), bad_terdeteksi=f"{int((h.rule_bad & yb).sum())}/{int(yb.sum())}",
                     good_lolos=f"{int((~h.rule_bad & ~yb).sum())}/{int((~yb).sum())}",
                     alarm_palsu_terbanyak=", ".join(f"{a}({n})" for a, n in alarm.most_common(2))))
print("\nPenilaian GOOD/BAD aturan di split test (video-level):")
display(pd.DataFrame(ring))

# ---- diagnosa: tag kesalahan QEVD mana yang paling sering LOLOS (video bad dinilai GOOD) ----
bq = hasil_te[(hasil_te.kualitas == "bad") & (hasil_te.sumber == "qevd")]
diag = []
for k, g in bq.groupby("kelas"):
    for t in set(x for ts in g.tags for x in ts) & TAG_BAD.get(k, set()):
        gt = g[g.tags.map(lambda ts: t in ts)]
        diag.append(dict(kelas=k, tag=t, video_bad=len(gt), tertangkap=int(gt.rule_bad.sum()),
                         ada_aturannya=any(t in v for v in TAG_ATURAN.values())))
if diag:
    diag = pd.DataFrame(diag).assign(porsi=lambda d: (d.tertangkap / d.video_bad).round(2))
    print("Recall per tag kesalahan (test). ada_aturannya=False berarti memang belum ada aturan untuk tag itu:")
    display(diag.sort_values(["kelas", "video_bad"], ascending=[True, False]))

## 9. Model kualitas ML per kelas (dibandingkan dengan aturan)

Untuk tiap kelas yang punya cukup video good **dan** bad, dilatih Random Forest good/bad. Mode **hybrid** (BAD kalau aturan **atau** ML bilang BAD) hanya dipakai kalau di test menaikkan balanced accuracy minimal 0.05 dibanding aturan saja. Kalau tidak, kelas itu tetap memakai aturan saja, yang alasannya selalu jelas.

In [ ]:
Y_Q = (META.kualitas == "good").astype(int).to_numpy()
mode_kualitas, ring_q = {}, []
for k in KELAS:
    mk = (META.kelas == k).to_numpy() & LAB
    vtr = VID[(VID.kelas == k) & (VID.split == "train") & VID.kualitas.isin(["good", "bad"])]
    n_good, n_bad = int((vtr.kualitas == "good").sum()), int((vtr.kualitas == "bad").sum())
    h = hasil_te[hasil_te.kelas == k]
    yb = (h.kualitas == "bad").to_numpy()
    row = dict(kelas=k, train_good=n_good, train_bad=n_bad, aturan=ba(yb, h.rule_bad), ml=np.nan, hybrid=np.nan)
    mode = "aturan"
    if n_good >= MIN_VIDEO_ML_KUALITAS and n_bad >= MIN_VIDEO_ML_KUALITAS and len(h):
        sel = tr & mk
        m = make_model(max_depth=9, n=300).fit(F_ALL[sel], Y_Q[sel], sample_weight=bobot_sampel(META[sel], "kualitas"))
        st = te & mk
        kol = list(m.classes_).index(1) if 1 in m.classes_ else None
        p = m.predict_proba(F_ALL[st])[:, kol] if kol is not None else np.zeros(st.sum())
        pv = pd.Series(p).groupby(META.video_id.to_numpy()[st]).mean().reindex(h.index)
        ml_bad = (pv < 0.5).to_numpy()
        row.update(ml=ba(yb, ml_bad), hybrid=ba(yb, h.rule_bad.to_numpy() | ml_bad))
        cukup = yb.sum() >= 5 and (~yb).sum() >= 5
        if cukup and np.isfinite(row["hybrid"]) and row["hybrid"] >= (row["aturan"] if np.isfinite(row["aturan"]) else 0) + 0.05:
            mode = "hybrid"
    row["mode_dipakai"] = mode
    mode_kualitas[k] = mode
    ring_q.append(row)

print("Balanced accuracy GOOD/BAD di test (0.5 = tebak-tebakan):")
display(pd.DataFrame(ring_q))
print("mode per kelas:", mode_kualitas)

## 10. Model final + simpan

Model dilatih ulang dengan **semua** data (train + test), dan ambang aturan dikalibrasi ulang dengan semua video. Hasilnya:
- `movit_model_v6.pkl` (di folder yang sama dengan `movit_fitur.py`): paket lengkap untuk `realtime_workout.py`;
- `aturan_v6.json`: ambang aturan yang bisa dibaca manusia, juga berguna kalau nanti aturannya di-port ke Kotlin.

In [ ]:
ATURAN_FINAL, lap_final = kalibrasi(VID, None, FPR_KELAS)
print("Ambang final (semua video):")
display(lap_final[["kelas", "aturan", "metrik", "arah", "default"] +
                  [c for c in ("n_good", "ambang_baru", "auc", "tpr", "fpr_good", "status") if c in lap_final.columns]])

final_kelas = make_model().fit(F_ALL, Y_KELAS, sample_weight=W_KELAS)
model_kualitas = {}
for k, mode in mode_kualitas.items():
    if mode == "hybrid":
        sel = (META.kelas == k).to_numpy() & LAB
        model_kualitas[k] = make_model(max_depth=9, n=300).fit(F_ALL[sel], Y_Q[sel],
                                                               sample_weight=bobot_sampel(META[sel], "kualitas"))

paket = dict(fitur_versi=FITUR_VERSI, kelas=KELAS, window=WINDOW, stride=STRIDE, target_fps=TARGET_FPS,
             cols=FEAT_COLS, model_kelas=final_kelas, model_kualitas=model_kualitas, mode_kualitas=mode_kualitas,
             aturan=ATURAN_FINAL, proporsi_window_bad=PROPORSI_WINDOW_BAD)
joblib.dump(paket, MODEL_DIR / "movit_model_v6.pkl")
(MODEL_DIR / "aturan_v6.json").write_text(json.dumps(
    {"fitur_versi": FITUR_VERSI, "mode_kualitas": mode_kualitas, "aturan": ATURAN_FINAL}, indent=2), encoding="utf-8")
print(f"movit_model_v6.pkl <- {len(F_ALL)} window, {META.video_id.nunique()} video | model kualitas ML: {list(model_kualitas)}")

y_kual = np.where(META.kualitas == "good", 1, np.where(META.kualitas == "bad", 0, -1))
np.savez_compressed(ROOT / "dataset_final_v6.npz", X=X_all, y_kelas=Y_KELAS, y_kualitas=y_kual,
                    split=META.split.to_numpy(dtype=str), sumber=META.sumber.to_numpy(dtype=str),
                    video_id=META.video_id.to_numpy(dtype=str), kelas_names=np.array(KELAS))
META.drop(columns=["group"]).assign(tags=META.tags.map(" | ".join)).to_csv(ROOT / "dataset_windows_v6.csv", index=False)
print(f"tersimpan: {MODEL_DIR / 'movit_model_v6.pkl'}, {MODEL_DIR / 'aturan_v6.json'}, "
      f"dataset_final_v6.npz + dataset_windows_v6.csv di {ROOT}")

## 11. Coba satu video: GOOD / BAD + alasan

Sama dengan yang dipakai realtime: aturan dari paket v6, ditambah model kualitas ML untuk kelas yang mode-nya `hybrid`. Output tanpa persen.

In [ ]:
def predict_video(path, gerakan=None):
    """gerakan: isi nama kelas (mis. "Pushup") kalau sudah tahu gerakannya; None = ditebak model."""
    paket = joblib.load(MODEL_DIR / "movit_model_v6.pkl")
    ws = make_windows(video_to_keypoints(path))
    if not ws:
        return {"file": Path(path).name, "form": "-", "alasan": ["Video terlalu pendek atau tubuh tidak terlihat."]}
    X = np.stack([w for _, w in ws])
    F, M = bangun_fitur(X), metrik_window(X)
    if gerakan is None:
        pk = paket["model_kelas"].predict(F)
        kelas = paket["kelas"][int(pd.Series(pk).mode().iloc[0])]
    else:
        kelas = gerakan

    hit = Counter()
    for i in range(len(X)):
        hit.update(p for _, p in cek_aturan(metrik_satu(M, i), kelas, paket["aturan"]))
    alasan = [p for p, n in hit.most_common() if n / len(X) >= paket["proporsi_window_bad"]]
    bad = bool(alasan)

    mq = paket["model_kualitas"].get(kelas)
    if mq is not None and 1 in mq.classes_:
        if mq.predict_proba(F)[:, list(mq.classes_).index(1)].mean() < 0.5:
            bad = True
    if bad and not alasan:
        alasan = [p for p, _ in hit.most_common(2)] or [pesan(kelas, "ML_BURUK")]

    hasil = {"file": Path(path).name, "gerakan": NAMA_TAMPIL[kelas], "form": "BAD" if bad else "GOOD",
             "alasan": alasan if bad else []}
    print(f"[{hasil['file']}] {hasil['gerakan']} -> {hasil['form']}")
    for a in hasil["alasan"]:
        print("   -", a)
    return hasil


# predict_video("contoh.mp4")
# predict_video("contoh.mp4", gerakan="Pushup")